In [4]:
%pip install kafka-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import sys

# Remove old Spark 3.5.9 configuration
os.environ.pop("SPARK_HOME", None)

# Tell PySpark to use the current Python 3.12
os.environ["PYSPARK_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))

Python: c:\Users\bda\AppData\Local\Programs\Python\Python312\python.exe
SPARK_HOME: None
PYSPARK_PYTHON: c:\Users\bda\AppData\Local\Programs\Python\Python312\python.exe
PYSPARK_DRIVER_PYTHON: jupyter


In [2]:
import pyspark
import os
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Kafka-Streaming').getOrCreate()


c:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
# Read Stream from Kafka

from pyspark.sql.functions import *

df_raw = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", "localhost:9092")
         .option("subscribe", "uber-rides")
         .load()
)

# Kafka gives key/value as binary → convert to string
df = df_raw.selectExpr("CAST(value AS STRING)")


AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

Schema tells Spark the structure of the JSON message coming from Kafka. To properly parse JSON into columns, Spark must know what fields exist and what their data types are. from_json() parses the JSON string into a structured object (StructType)   something like: schema = StructType([
    StructField("ride_id", LongType()),
    StructField("city", StringType()),
    StructField("timestamp", StringType()),
    StructField("distance_km", DoubleType()),
    StructField("fare_usd", DoubleType())
])


.alias("data") 
Names the parsed struct column as data
data
   ride_id
   city
   timestamp
   distance_km
   fare_usd

.select("data.*")
This expands the struct into flat columns.
input in the form: data (struct<ride_id, city, timestamp, ...>)
output generated is:
| ride_id | city | timestamp  | distance_km | fare_usd |
| ------- | ---- | ---------- | ----------- | -------- |
| 101     | NYC  | 2025-01-01 | 4.3         | 12.5     |



In [ ]:
schema = """
ride_id LONG,
city STRING,
timestamp STRING,
distance_km DOUBLE,
fare_usd DOUBLE
"""

df_parsed = df.select(from_json(col("value"), schema).alias("data")).select("data.*")


In [ ]:
# get number of trips done for every minute

trips_per_min = (
    df_parsed
        .withColumn("event_time", to_timestamp("timestamp"))
        .groupBy(window(col("event_time"), "1 minute"))
        .count()
)


In [ ]:
#Average ride price per city

#It computes average ride fare per city, over a 1-minute rolling event-time window, 
#from streaming Kafka data.
avg_fare = (
    df_parsed
        .withColumn("event_time", to_timestamp("timestamp"))
        .groupBy(
            window(col("event_time"), "1 minute"),
            col("city")
        )
        .agg(avg("fare_usd").alias("avg_fare"))
)


In [ ]:
# Computing trips per minute

query1 = (
    trips_per_min.writeStream
        .format("console")
        .outputMode("complete")
        .option("truncate", False)
        .start()
)


In [ ]:
# Computing average fare per city

query2 = (
    avg_fare.writeStream
        .format("console")
        .outputMode("complete")
        .option("truncate", False)
        .start()
)


In [ ]:
query1.stop()
query2.stop()